###Load raw to struct

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

####Products

In [0]:
dbutils.widgets.text("pipeline_run_id","")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
ingestion_date = current_timestamp()

In [0]:
# Read bronze tables
raw_products = spark.table("ecom_ref_raw.products")
raw_category = spark.table("ecom_ref_raw.product_category")

# Join products with category
products = raw_products.join(
    raw_category,
    on="product_category_name",
    how="left"
)

# Calculate averages for null replacement
avg_weight = products.agg(avg("product_weight_g")).collect()[0][0]
avg_length = products.agg(avg("product_length_cm")).collect()[0][0]
avg_height = products.agg(avg("product_height_cm")).collect()[0][0]
avg_width  = products.agg(avg("product_width_cm")).collect()[0][0]

# Cleaning
products = (products
                    .fillna({
                            "product_category_name_english": "Unknown",
                            "product_name_lenght": 0,
                            "product_description_lenght": 0,
                            "product_photos_qty": 0
                        })
                    .withColumn("product_weight_g",coalesce(col("product_weight_g"), lit(avg_weight)))
                    .withColumn("product_length_cm",coalesce(col("product_length_cm"), lit(avg_length)))
                    .withColumn("product_height_cm",coalesce(col("product_height_cm"), lit(avg_height)))
                    .withColumn("product_width_cm",coalesce(col("product_width_cm"), lit(avg_width)))
                    .withColumn("product_weight_g",col("product_weight_g").cast(DoubleType()))
                    .withColumn("product_length_cm",col("product_length_cm").cast(DoubleType()))
                    .withColumn("product_height_cm",col("product_height_cm").cast(DoubleType()))
                    .withColumn("product_width_cm",col("product_width_cm").cast(DoubleType()))
                    .withColumn("product_photos_qty",col("product_photos_qty").cast(IntegerType()))
                    .withColumn("product_name_length",col("product_name_lenght").cast(IntegerType()))
                    .withColumn("product_description_length",col("product_description_lenght").cast(IntegerType()))
            )

# Remove duplicates
products = products.dropDuplicates(["product_id"])

# Remove null PKs
products = products.filter(col("product_id").isNotNull())

# Add tracking columns
products = (products.withColumn("ingestion_date_struct", to_timestamp(lit(ingestion_date)))
                    .withColumn("pipeline_run_id_struct", lit(pipeline_run_id))
    )

products.createOrReplaceTempView("products")

In [0]:
# Data quality check
total = products.count()
print(f"products total records: {total}")

In [0]:
spark.sql(f'''
MERGE INTO ecom_ref_struct.products AS target
USING products AS source
ON target.product_id = source.product_id    
WHEN MATCHED THEN UPDATE SET
    target.product_category_name = source.product_category_name,
    target.product_category_name_english = source.product_category_name_english,
    target.product_name_length = source.product_name_length,
    target.product_description_length = source.product_description_length,
    target.product_photos_qty = source.product_photos_qty,
    target.product_weight_g = source.product_weight_g,
    target.product_length_cm = source.product_length_cm,
    target.product_height_cm = source.product_height_cm,
    target.product_width_cm = source.product_width_cm,
    target.ingestion_date = source.ingestion_date_struct,
    target.pipeline_run_id = source.pipeline_run_id_struct
WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_category_name,
    product_category_name_english,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm,
    ingestion_date,
    pipeline_run_id
) VALUES (
    source.product_id, 
    source.product_category_name, 
    source.product_category_name_english, 
    source.product_name_length, 
    source.product_description_length, 
    source.product_photos_qty, 
    source.product_weight_g, 
    source.product_length_cm, 
    source.product_height_cm, 
    source.product_width_cm, 
    source.ingestion_date_struct, 
    source.pipeline_run_id_struct
)

''').display()

####Geolocation

In [0]:
# Read bronze
raw_geo = spark.table("ecom_ref_raw.geolocation")

# Cleaning
geo = (raw_geo.withColumn("geolocation_lat",col("geolocation_lat").cast(DoubleType()))
                 .withColumn("geolocation_lng",col("geolocation_lng").cast(DoubleType()))
                 .withColumn("geolocation_city",lower(trim(regexp_replace(col("geolocation_city"), "[^a-zA-Z\\s]", ""))))
                 .withColumn("geolocation_state",upper(trim(col("geolocation_state"))))
        )

# Validate Brazil lat/lng ranges
geo = geo.filter(
    (col("geolocation_lat").between(-33, 5)) &
    (col("geolocation_lng").between(-73, -35))
)

# Remove nulls
geo = geo.filter(
    col("geolocation_zip_code_prefix").isNotNull() &
    col("geolocation_lat").isNotNull() &
    col("geolocation_lng").isNotNull()
)

# Remove duplicates on zip code
geo = geo.dropDuplicates(["geolocation_zip_code_prefix"])

# Add tracking columns
geo = (geo.withColumn("ingestion_date_struct", to_timestamp(lit(ingestion_date)))
                    .withColumn("pipeline_run_id_struct", lit(pipeline_run_id))
    )

geo = geo.drop("ingestion_date","pipeline_run_id")

geo.createOrReplaceTempView("geolocation")

In [0]:
# Data quality check
total = geo.count()
print(f"geolocation total records: {total}")

In [0]:
spark.sql(f'''
MERGE INTO ecom_ref_struct.geolocation AS target
USING geolocation AS source
ON target.geolocation_zip_code_prefix = source.geolocation_zip_code_prefix    
WHEN MATCHED THEN UPDATE SET
    target.geolocation_lat = source.geolocation_lat,
    target.geolocation_lng = source.geolocation_lng,
    target.geolocation_city = source.geolocation_city,
    target.geolocation_state = source.geolocation_state,
    target.ingestion_date = source.ingestion_date_struct,
    target.pipeline_run_id = source.pipeline_run_id_struct
WHEN NOT MATCHED THEN INSERT (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state,
    ingestion_date,
    pipeline_run_id
) VALUES (
    source.geolocation_zip_code_prefix,
    source.geolocation_lat,
    source.geolocation_lng,
    source.geolocation_city,
    source.geolocation_state,
    source.ingestion_date_struct,
    source.pipeline_run_id_struct
)
''').display()